# HydroTwin LSTM Model

This notebook develops Long Short Term Memory models for the Turbine Guide Bearing and Upper Guide Bearing.

Each model uses 36 historical observations of four input variables, representing six hours of operating history.

The model predicts the corresponding guide bearing vibration at the following 10 minute interval.

September data is used for validation during model development. The November final test dataset remains untouched until all candidate models have been finalized.

## 1. Import Libraries

In [ ]:
from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score
)

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

In [ ]:
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
PROJECT_ROOT = Path("..").resolve()

PREPARED_DIR = (
    PROJECT_ROOT /
    "data" /
    "prepared"
)

TGB_DIR = PREPARED_DIR / "tgb"
UGB_DIR = PREPARED_DIR / "ugb"

MODEL_DIR = (
    PROJECT_ROOT /
    "models" /
    "lstm"
)

SCALER_DIR = (
    MODEL_DIR /
    "scalers"
)

RESULTS_DIR = (
    PROJECT_ROOT /
    "results"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SCALER_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(PROJECT_ROOT)

In [ ]:
X_train_tgb = np.load(
    TGB_DIR / "X_train.npy"
)

y_train_tgb = np.load(
    TGB_DIR / "y_train.npy"
)

X_val_tgb = np.load(
    TGB_DIR / "X_val.npy"
)

y_val_tgb = np.load(
    TGB_DIR / "y_val.npy"
)

In [ ]:
X_train_ugb = np.load(
    UGB_DIR / "X_train.npy"
)

y_train_ugb = np.load(
    UGB_DIR / "y_train.npy"
)

X_val_ugb = np.load(
    UGB_DIR / "X_val.npy"
)

y_val_ugb = np.load(
    UGB_DIR / "y_val.npy"
)

In [ ]:
print("TGB:", X_train_tgb.shape, X_val_tgb.shape)
print("UGB:", X_train_ugb.shape, X_val_ugb.shape)

## 2. TGB Data Scaling

MinMax scaling is fitted using only the training dataset to prevent information leakage.

The four input features are scaled individually across all training time steps. The fitted scaler is then applied to the September validation dataset.

In [ ]:
tgb_feature_scaler = MinMaxScaler()

X_train_tgb_2d = X_train_tgb.reshape(
    -1,
    X_train_tgb.shape[-1]
)

X_val_tgb_2d = X_val_tgb.reshape(
    -1,
    X_val_tgb.shape[-1]
)

tgb_feature_scaler.fit(
    X_train_tgb_2d
)

X_train_tgb_scaled = (
    tgb_feature_scaler
    .transform(X_train_tgb_2d)
    .reshape(X_train_tgb.shape)
)

X_val_tgb_scaled = (
    tgb_feature_scaler
    .transform(X_val_tgb_2d)
    .reshape(X_val_tgb.shape)
)

In [ ]:
print(
    "TGB scaled training shape:",
    X_train_tgb_scaled.shape
)

print(
    "TGB scaled validation shape:",
    X_val_tgb_scaled.shape
)

print(
    "Training minimum:",
    X_train_tgb_scaled.min()
)

print(
    "Training maximum:",
    X_train_tgb_scaled.max()
)

In [ ]:
tgb_target_scaler = MinMaxScaler()

y_train_tgb_scaled = (
    tgb_target_scaler
    .fit_transform(
        y_train_tgb.reshape(-1, 1)
    )
    .ravel()
)

y_val_tgb_scaled = (
    tgb_target_scaler
    .transform(
        y_val_tgb.reshape(-1, 1)
    )
    .ravel()
)

In [ ]:
print(
    "Original TGB target range:",
    y_train_tgb.min(),
    "to",
    y_train_tgb.max()
)

print(
    "Scaled TGB target range:",
    y_train_tgb_scaled.min(),
    "to",
    y_train_tgb_scaled.max()
)

In [ ]:
joblib.dump(
    tgb_feature_scaler,
    SCALER_DIR / "tgb_feature_scaler.pkl"
)

joblib.dump(
    tgb_target_scaler,
    SCALER_DIR / "tgb_target_scaler.pkl"
)

print(
    "TGB scalers saved."
)

## 3. TGB LSTM Model

A single layer LSTM architecture is used to model the temporal behaviour of the Turbine Guide Bearing.

The model receives 36 historical observations with four input features and predicts the vibration value at the next 10 minute interval.

In [ ]:
tgb_lstm_model = Sequential([
    Input(
        shape=(
            X_train_tgb_scaled.shape[1],
            X_train_tgb_scaled.shape[2]
        )
    ),

    LSTM(
        64
    ),

    Dropout(
        0.2
    ),

    Dense(
        1
    )
])

In [ ]:
tgb_lstm_model.compile(
    optimizer="adam",
    loss="mse"
)

In [ ]:
tgb_lstm_model.summary()

In [ ]:
tgb_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
    verbose=1
)

tgb_reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

## 4. TGB Model Training

The LSTM is trained using June to August data. September is used exclusively for validation and early stopping.

In [ ]:
tgb_history = tgb_lstm_model.fit(
    X_train_tgb_scaled,
    y_train_tgb_scaled,

    validation_data=(
        X_val_tgb_scaled,
        y_val_tgb_scaled
    ),

    epochs=100,
    batch_size=32,

    callbacks=[
        tgb_early_stopping,
        tgb_reduce_lr
    ],

    verbose=1
)

In [ ]:
tgb_epochs = len(
    tgb_history.history["loss"]
)

print(
    "TGB epochs completed:",
    tgb_epochs
)

In [ ]:
plt.figure(
    figsize=(9, 5)
)

plt.plot(
    tgb_history.history["loss"],
    label="Training Loss"
)

plt.plot(
    tgb_history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")

plt.title(
    "TGB LSTM Training History"
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
tgb_val_pred_scaled = (
    tgb_lstm_model.predict(
        X_val_tgb_scaled,
        verbose=0
    )
    .ravel()
)

In [ ]:
tgb_val_pred = (
    tgb_target_scaler
    .inverse_transform(
        tgb_val_pred_scaled.reshape(-1, 1)
    )
    .ravel()
)

In [ ]:
print(
    "Prediction shape:",
    tgb_val_pred.shape
)

print(
    "Actual shape:",
    y_val_tgb.shape
)

In [ ]:
tgb_lstm_rmse = np.sqrt(
    mean_squared_error(
        y_val_tgb,
        tgb_val_pred
    )
)

tgb_lstm_mae = mean_absolute_error(
    y_val_tgb,
    tgb_val_pred
)

tgb_lstm_mape = (
    mean_absolute_percentage_error(
        y_val_tgb,
        tgb_val_pred
    )
    * 100
)

tgb_lstm_r2 = r2_score(
    y_val_tgb,
    tgb_val_pred
)

In [ ]:
print(
    f"TGB LSTM Validation RMSE: "
    f"{tgb_lstm_rmse:.4f}"
)

print(
    f"TGB LSTM Validation MAE: "
    f"{tgb_lstm_mae:.4f}"
)

print(
    f"TGB LSTM Validation MAPE: "
    f"{tgb_lstm_mape:.2f}%"
)

print(
    f"TGB LSTM Validation R²: "
    f"{tgb_lstm_r2:.4f}"
)

In [ ]:
plot_points = 500

plt.figure(
    figsize=(14, 5)
)

plt.plot(
    y_val_tgb[-plot_points:],
    label="Actual TGB Vibration"
)

plt.plot(
    tgb_val_pred[-plot_points:],
    label="LSTM Prediction"
)

plt.xlabel(
    "Validation Observation"
)

plt.ylabel(
    "TGB Vibration"
)

plt.title(
    "TGB LSTM Validation Prediction"
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
tgb_lstm_model.save(
    MODEL_DIR /
    "tgb_lstm.keras"
)

print(
    "TGB LSTM saved:",
    (
        MODEL_DIR /
        "tgb_lstm.keras"
    ).exists()
)

## 5. UGB Data Scaling

Separate scalers are fitted for the Upper Guide Bearing using only the UGB training dataset.

In [ ]:
ugb_feature_scaler = MinMaxScaler()

X_train_ugb_2d = X_train_ugb.reshape(
    -1,
    X_train_ugb.shape[-1]
)

X_val_ugb_2d = X_val_ugb.reshape(
    -1,
    X_val_ugb.shape[-1]
)

ugb_feature_scaler.fit(
    X_train_ugb_2d
)

X_train_ugb_scaled = (
    ugb_feature_scaler
    .transform(X_train_ugb_2d)
    .reshape(X_train_ugb.shape)
)

X_val_ugb_scaled = (
    ugb_feature_scaler
    .transform(X_val_ugb_2d)
    .reshape(X_val_ugb.shape)
)

In [ ]:
ugb_target_scaler = MinMaxScaler()

y_train_ugb_scaled = (
    ugb_target_scaler
    .fit_transform(
        y_train_ugb.reshape(-1, 1)
    )
    .ravel()
)

y_val_ugb_scaled = (
    ugb_target_scaler
    .transform(
        y_val_ugb.reshape(-1, 1)
    )
    .ravel()
)

In [ ]:
joblib.dump(
    ugb_feature_scaler,
    SCALER_DIR / "ugb_feature_scaler.pkl"
)

joblib.dump(
    ugb_target_scaler,
    SCALER_DIR / "ugb_target_scaler.pkl"
)

print(
    "UGB scalers saved."
)

In [ ]:
ugb_lstm_model = Sequential([
    Input(
        shape=(
            X_train_ugb_scaled.shape[1],
            X_train_ugb_scaled.shape[2]
        )
    ),

    LSTM(
        64
    ),

    Dropout(
        0.2
    ),

    Dense(
        1
    )
])

In [ ]:
ugb_lstm_model.compile(
    optimizer="adam",
    loss="mse"
)

In [ ]:
ugb_lstm_model.summary()

In [ ]:
ugb_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
    verbose=1
)

ugb_reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

In [ ]:
ugb_history = ugb_lstm_model.fit(
    X_train_ugb_scaled,
    y_train_ugb_scaled,

    validation_data=(
        X_val_ugb_scaled,
        y_val_ugb_scaled
    ),

    epochs=100,
    batch_size=32,

    callbacks=[
        ugb_early_stopping,
        ugb_reduce_lr
    ],

    verbose=1
)

In [ ]:
ugb_val_pred_scaled = (
    ugb_lstm_model.predict(
        X_val_ugb_scaled,
        verbose=0
    )
    .ravel()
)

ugb_val_pred = (
    ugb_target_scaler
    .inverse_transform(
        ugb_val_pred_scaled.reshape(-1, 1)
    )
    .ravel()
)

In [ ]:
ugb_lstm_rmse = np.sqrt(
    mean_squared_error(
        y_val_ugb,
        ugb_val_pred
    )
)

ugb_lstm_mae = mean_absolute_error(
    y_val_ugb,
    ugb_val_pred
)

ugb_lstm_mape = (
    mean_absolute_percentage_error(
        y_val_ugb,
        ugb_val_pred
    )
    * 100
)

ugb_lstm_r2 = r2_score(
    y_val_ugb,
    ugb_val_pred
)

In [ ]:
print(
    f"UGB LSTM Validation RMSE: "
    f"{ugb_lstm_rmse:.4f}"
)

print(
    f"UGB LSTM Validation MAE: "
    f"{ugb_lstm_mae:.4f}"
)

print(
    f"UGB LSTM Validation MAPE: "
    f"{ugb_lstm_mape:.2f}%"
)

print(
    f"UGB LSTM Validation R²: "
    f"{ugb_lstm_r2:.4f}"
)

In [ ]:
ugb_lstm_model.save(
    MODEL_DIR /
    "ugb_lstm.keras"
)

print(
    "UGB LSTM saved:",
    (
        MODEL_DIR /
        "ugb_lstm.keras"
    ).exists()
)

In [ ]:
lstm_validation_results = pd.DataFrame({
    "Component": [
        "TGB",
        "UGB"
    ],

    "Model": [
        "LSTM",
        "LSTM"
    ],

    "RMSE": [
        tgb_lstm_rmse,
        ugb_lstm_rmse
    ],

    "MAE": [
        tgb_lstm_mae,
        ugb_lstm_mae
    ],

    "MAPE": [
        tgb_lstm_mape,
        ugb_lstm_mape
    ],

    "R2": [
        tgb_lstm_r2,
        ugb_lstm_r2
    ],

    "Epochs": [
        len(
            tgb_history.history["loss"]
        ),
        len(
            ugb_history.history["loss"]
        )
    ]
})

lstm_validation_results

In [ ]:
lstm_validation_results.to_csv(
    RESULTS_DIR /
    "lstm_validation_results.csv",
    index=False
)